# Time Series Classification: Structural Pattern Recognition

In many high-stakes environments, the pressing question is not *"what will the value be at time t+1?"* but *"what state does this sequence represent?"* This shift from continuous prediction to discrete categorization moves us from regression losses to categorical cross-entropy, and from lag engineering to structural pattern recognition.

In this notebook we apply the **ROCKET** family to Human Activity Recognition (HAR). We use the `BasicMotions` dataset as a proxy: 6 sensor channels (3-axial acceleration and 3-axial angular velocity) and four activity classes (Badminton, Running, Standing, Walking). We build a `MiniRocketMultivariate` + `RidgeClassifierCV` pipeline, which trains in seconds on a CPU while reaching accuracy comparable to HIVE-COTE.

## Load the dataset

`BasicMotions` ships with `sktime` as a nested-pandas panel. Each row is a sample, each column is a sensor channel, and each cell holds the full time series for that channel. Note the 3D shape `(n_samples, n_vars, n_timepoints)` when we convert to NumPy — we are not dealing with a flat tabular dataset.

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sktime.datasets import load_basic_motions

# Load Multivariate HAR Data (BasicMotions)
X, y = load_basic_motions(return_X_y=True)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Samples: {len(X_train)} train / {len(X_test)} test")
print(f"Variables (channels): {X_train.shape[1]}")
print(f"Timepoints per sample: {len(X_train.iloc[0, 0])}")
print(f"Classes: {np.unique(y)}")

Samples: 56 train / 24 test
Variables (channels): 6
Timepoints per sample: 100
Classes: ['badminton' 'running' 'standing' 'walking']


## Build the MiniRocket pipeline

`MiniRocket` projects each series into a high-dimensional feature space (10,000 features) using random convolutional kernels. Because the projection is so wide, a **linear** classifier is usually enough to separate the classes — we use `RidgeClassifierCV`, which tunes the L2 penalty via cross-validation. This combination is orders of magnitude faster than training an RNN.

In [2]:
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sktime.transformations.panel.rocket import MiniRocketMultivariate

# MiniRocket transforms the time series into 10,000 features
# RidgeClassifierCV learns the linear boundary with cross-validation
classifier = make_pipeline(
    MiniRocketMultivariate(num_kernels=10_000, random_state=42),
    StandardScaler(with_mean=False),
    RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)),
)

print("Fitting pipeline...")
classifier.fit(X_train, y_train)

Fitting pipeline...


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('minirocketmultivariate', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",False
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.",array([1.0000...00000000e+03])
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`accuracy ` otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding

## Evaluate the model

Accuracy on the test set gives the headline number, but the classification report is what matters in practice. In HAR, models commonly confuse *Standing* and *Sitting* (both static signatures) while easily separating dynamic activities like *Running*.

In [3]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

   badminton       1.00      1.00      1.00         6
     running       1.00      1.00      1.00         4
    standing       1.00      1.00      1.00         8
     walking       1.00      1.00      1.00         6

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



## Summary of methods

The choice of algorithm depends heavily on deployment constraints:

| Algorithm    | Type          | Accuracy (Rank) | Training Speed | Best Use Case |
|--------------|---------------|-----------------|----------------|----------------|
| HIVE-COTE 2.0| Meta-Ensemble | 1 (Best)        | Very Slow      | Offline diagnosis (e.g., medical) where max accuracy is critical |
| MultiRocket  | Convolutional | ~1.5            | Fast           | The default choice. High accuracy, deployable on standard hardware |
| InceptionTime| Deep CNN      | ~3              | Medium (GPU)   | Massive datasets (>1M samples) or generative tasks |
| 1-NN DTW     | Distance      | ~5              | Slow $O(N^2)$  | Small datasets (<1k samples); robust baseline |

For most industrial applications, **MiniRocket** or **MultiRocket** hits the efficient frontier — near-SOTA accuracy at a training cost low enough for frequent retraining.

If you hit performance bottlenecks, `aeon` (a specialist fork of `sktime` focused on classification/clustering) provides Numba-optimized drop-in replacements such as `aeon.classification.convolution_based.RocketClassifier`.